# nixl_cu13 smoke test

NVIDIA NIXL moves memory between agents through pluggable back ends.

Examples 1-4 need no GPU and no network: two agents inside this one process talk through UCX's same-machine transports. Example 5 moves GPU memory and is skipped where there is no GPU.

## Example 1 — the compiled module imports and an agent lists its back ends


In [ ]:
import importlib.metadata as metadata, importlib.util
from nixl_cu13._api import nixl_agent, nixl_agent_config
import nixl_cu13._bindings as bindings, nixl_cu13._utils as utils
assert bindings.__name__ == 'nixl_cu13._bindings' and utils.__name__ == 'nixl_cu13._utils'
has_nixl_ep = importlib.util.find_spec('nixl_ep_cu13') is not None
agent = nixl_agent('nb-loader', nixl_agent_config(backends=[]))  # no back end started yet
plugins = sorted(agent.get_plugin_list())
assert {'UCX', 'POSIX'} <= set(plugins), plugins
assert 'DRAM_SEG' in agent.get_plugin_mem_types('UCX') and 'FILE_SEG' in agent.get_plugin_mem_types('POSIX')
if 'OBJ' in plugins:
    assert 'OBJ_SEG' in agent.get_plugin_mem_types('OBJ')
agent.create_backend('UCX')
assert 'UCX' in agent.backends and 'DRAM_SEG' in agent.get_backend_mem_types('UCX')
print(f'OK -- nixl-cu13 {metadata.version("nixl-cu13")}: back ends {plugins}; '
      f'NIXL-EP module: {"installed" if has_nixl_ep else "not installed"}')

## Example 2 — one agent reads 1 MiB out of another agent's memory through UCX


In [ ]:
# Adapted from NIXL's examples/python/basic_two_peers.py (v1.4.1, Apache-2.0): both peers live in this one process.
import time, uuid, torch
from nixl_cu13._api import nixl_agent, nixl_agent_config
SIZE = 1 << 20
target = nixl_agent(f'target-{uuid.uuid4()}', nixl_agent_config(backends=['UCX']))
initiator = nixl_agent(f'initiator-{uuid.uuid4()}', nixl_agent_config(backends=['UCX']))
source = (torch.arange(SIZE) % 251).to(torch.uint8)
sink = torch.zeros(SIZE, dtype=torch.uint8)
target_registration, initiator_registration = target.register_memory(source), initiator.register_memory(sink)
# Metadata is exchanged AFTER registration: it carries what each agent registered.
initiator.add_remote_agent(target.get_agent_metadata())
target.add_remote_agent(initiator.get_agent_metadata())
handle = initiator.initialize_xfer('READ', initiator.get_xfer_descs(sink), target.get_xfer_descs(source), target.name)
state, deadline = initiator.transfer(handle), time.monotonic() + 60
while state == 'PROC' and time.monotonic() < deadline:
    state = initiator.check_xfer_state(handle)
assert state == 'DONE', state
assert torch.equal(sink, source), 'the bytes read do not match the bytes the target holds'
assert initiator.query_xfer_backend(handle) == 'UCX', initiator.query_xfer_backend(handle)
initiator.release_xfer_handle(handle)
initiator.deregister_memory(initiator_registration); target.deregister_memory(target_registration)
print(f'OK -- READ of {SIZE} bytes between two agents through UCX, byte for byte')


## Example 3 — the file back end (POSIX) writes memory to a file and reads it back


In [ ]:
# Adapted from NIXL's examples/python/nixl_gds_example.py (v1.4.1, Apache-2.0), with the POSIX back end in place of GDS.
import os, tempfile, time, torch
from nixl_cu13._api import nixl_agent, nixl_agent_config
SIZE = 1 << 20
agent = nixl_agent('nb-posix', nixl_agent_config(backends=['POSIX']))
source, sink = (torch.arange(SIZE) % 251).to(torch.uint8), torch.zeros(SIZE, dtype=torch.uint8)
memory = agent.register_memory([source, sink])
path = os.path.join(tempfile.mkdtemp(prefix='nixl_nb_'), 'block.bin')
fd = os.open(path, os.O_RDWR | os.O_CREAT)
file_registration = agent.register_memory([(0, SIZE, fd, 'block')], 'FILE')
for operation, buffer in (('WRITE', source), ('READ', sink)):
    handle = agent.initialize_xfer(operation, agent.get_xfer_descs(buffer), file_registration.trim(), agent.name)
    state, deadline = agent.transfer(handle), time.monotonic() + 60
    while state == 'PROC' and time.monotonic() < deadline:
        state = agent.check_xfer_state(handle)
    assert state == 'DONE', (operation, state)
    agent.release_xfer_handle(handle)
assert os.path.getsize(path) == SIZE, os.path.getsize(path)
assert open(path, 'rb').read() == bytes(source.numpy()), 'the file does not hold the bytes that were written'
assert torch.equal(sink, source), 'the bytes read back differ from the bytes written'
agent.deregister_memory(memory); agent.deregister_memory(file_registration); os.close(fd)
print(f'OK -- POSIX back end wrote {SIZE} bytes to a file and read them back')


## Example 4 — descriptors survive being sent as bytes, and a WRITE tells the target it is done


In [ ]:
import time, uuid, torch
from nixl_cu13._api import nixl_agent, nixl_agent_config
target = nixl_agent(f'target-{uuid.uuid4()}', nixl_agent_config(backends=['UCX']))
initiator = nixl_agent(f'initiator-{uuid.uuid4()}', nixl_agent_config(backends=['UCX']))
rows = torch.arange(10 * 16, dtype=torch.float32).reshape(10, 16)  # ten separate blocks, like a KV cache's layers
sink = torch.zeros_like(rows)
initiator_registration, target_registration = initiator.register_memory(rows), target.register_memory(sink)
initiator.add_remote_agent(target.get_agent_metadata())
target.add_remote_agent(initiator.get_agent_metadata())
# Between two machines the target's descriptors travel as bytes; here they make the same round trip.
as_bytes = target.get_serialized_descs(target.get_xfer_descs([sink[i, :] for i in range(10)]))
remote = initiator.deserialize_descs(as_bytes)
assert isinstance(as_bytes, bytes) and remote.descCount() == 10, (type(as_bytes), remote.descCount())
local = initiator.get_xfer_descs([rows[i, :] for i in range(10)])
handle = initiator.initialize_xfer('WRITE', local, remote, target.name, b'ten-blocks-written')
state, deadline = initiator.transfer(handle), time.monotonic() + 60
while state == 'PROC' and time.monotonic() < deadline:
    state = initiator.check_xfer_state(handle)
assert state == 'DONE', state
notifications = {}
while initiator.name not in notifications and time.monotonic() < deadline:
    notifications = target.get_new_notifs()
assert notifications.get(initiator.name) == [b'ten-blocks-written'], notifications
assert torch.equal(sink, rows), 'the ten blocks did not arrive intact'
print('OK -- 10 blocks written through serialized descriptors; the target was told', notifications[initiator.name][0])


## Example 5 — GPU memory: GPU to GPU and GPU to host, through UCX's CUDA module


In [ ]:
import time, uuid, torch
from nixl_cu13._api import nixl_agent, nixl_agent_config
assert torch.cuda.is_available(), 'requires-gpu cell ran but torch sees no CUDA device'
SIZE = 1 << 20
def read(sink_device):
    target = nixl_agent(f'target-{uuid.uuid4()}', nixl_agent_config(backends=['UCX']))
    initiator = nixl_agent(f'initiator-{uuid.uuid4()}', nixl_agent_config(backends=['UCX']))
    source = (torch.arange(SIZE, device='cuda:0') % 251).to(torch.uint8)
    sink = torch.zeros(SIZE, dtype=torch.uint8, device=sink_device)
    registrations = target.register_memory(source), initiator.register_memory(sink)
    initiator.add_remote_agent(target.get_agent_metadata())
    target.add_remote_agent(initiator.get_agent_metadata())
    handle = initiator.initialize_xfer('READ', initiator.get_xfer_descs(sink), target.get_xfer_descs(source), target.name)
    state, deadline = initiator.transfer(handle), time.monotonic() + 60
    while state == 'PROC' and time.monotonic() < deadline:
        state = initiator.check_xfer_state(handle)
    assert state == 'DONE', (sink_device, state)
    assert torch.equal(sink.cpu(), source.cpu()), f'GPU -> {sink_device}: the bytes differ'
    initiator.release_xfer_handle(handle)
    target.deregister_memory(registrations[0]); initiator.deregister_memory(registrations[1])
assert 'VRAM_SEG' in nixl_agent('nb-gpu', nixl_agent_config(backends=['UCX'])).get_backend_mem_types('UCX')
read('cuda:0')
read('cpu')
print(f'OK -- {SIZE} bytes of GPU memory read GPU -> GPU and GPU -> host on', torch.cuda.get_device_name(0))
